# 04b — Inspección de splits temporales (`splits-0.1.0`)

Notebook **explicativo / de inspección** del paso de asignación de splits sobre ventanas `features-0.2.0`.

**Solo lectura.** No reasigna splits, no escribe `data/splits/` nuevos, no entrena, no genera sintéticos.
No muta `data/features/` ni `data/clean/`.

Fuente de verdad: artefactos (`splits-0.1.0`):
- `data/splits/window_splits_stride{1,10,30,65}.parquet`
- `data/splits/split_manifest.json` + `checksums.sha256`
- (join demo opcional) `data/features/windows_65_stride1.parquet`

### Cómo ejecutarlo

Desde la raíz del repo:

```bash
uv pip install --python .venv/bin/python -r requirements.txt matplotlib ipykernel
jupyter notebook notebooks/04b_inspect_splits.ipynb
```

Kernel del `.venv`. Detalle: `notebooks/README_04b.md`.


## Controles

| Control | Default | Uso |
|---|---|---|
| `STRIDE` | `1` | Parquet de splits a inspeccionar en profundidad (`primary_stride`) |
| `STRIDE_COMPARE` | `65` | Segundo stride para comparar composición / `nvda_visible` |
| `SAVE_FIGS` | `False` | Si `True`, exporta PNGs a `notebooks/figures/04b_*` |

`STRIDE=1` es el **default recomendado** comparable; existen splits para todos los strides del menú.


In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# --- Controles ---
STRIDE = 1              # primary / default recomendado
STRIDE_COMPARE = 65     # segundo stride (advertencia nvda_visible pequeño)
SAVE_FIGS = False

EXPECTED_VERSION = "splits-0.1.0"
EXPECTED_FEATURES_VERSION = "features-0.2.0"
EXPECTED_STRIDES = [1, 10, 30, 65]
PRIMARY_STRIDE = 1
SPLIT_ORDER = ["donor_train", "donor_val", "nvda_visible", "nvda_test", "unused"]
SPLIT_COLORS = {
    "donor_train": "#2E86AB",
    "donor_val": "#A23B72",
    "nvda_visible": "#F18F01",
    "nvda_test": "#C73E1D",
    "unused": "#6C757D",
}
WINDOWS_STRIDE1_SHA = "58bf4c4788cc4ae4feed14c2173419dea876a322c9e6f66d510c9dfb6c00bccf"

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / "data" / "splits").is_dir() else cwd.parent
SPLITS_DIR = ROOT / "data" / "splits"
FEATURES_DIR = ROOT / "data" / "features"
FIG_DIR = ROOT / "notebooks" / "figures"

assert SPLITS_DIR.is_dir(), f"No se encuentra data/splits en {ROOT}"
assert STRIDE in EXPECTED_STRIDES, f"STRIDE={STRIDE} no está en {EXPECTED_STRIDES}"
assert STRIDE_COMPARE in EXPECTED_STRIDES, f"STRIDE_COMPARE={STRIDE_COMPARE} no está en {EXPECTED_STRIDES}"
print(f"ROOT = {ROOT}")
print(f"STRIDE={STRIDE} | STRIDE_COMPARE={STRIDE_COMPARE} | SAVE_FIGS={SAVE_FIGS}")


## 1. Contexto: por qué existen estos splits

Los generadores (VAE / GAN / Diffusion) **no deben entrenar mirando NVDA de test**.
Los splits parten el menú de ventanas en roles mutuamente excluyentes, según `ticker` y
`window_end_date` (mismas reglas en todos los strides):

| Split | Quién | Para qué |
|---|---|---|
| `donor_train` | 10 donors, `window_end` ∈ [2012-01-01, 2021-12-31] | **Entrenar** el generador |
| `donor_val` | 10 donors, `window_end` ∈ [2022-01-01, 2022-12-31] | Checkpoint / early-stop / hiperparámetros |
| `nvda_visible` | NVDA, `window_end` ∈ [2022-07-01, 2022-12-31] | Inspección / calibración **fuera** del train del generador |
| `nvda_test` | NVDA, `window_end` ∈ [2023-01-01, 2025-12-31] | Hold-out final — **no mirar para tunear** |
| `unused` | Todo lo demás | Fuera del protocolo oficial |

```
data/features/windows_65_stride{N}.parquet
        │
        │  assign_splits (solo etiquetas; no toca features)
        ▼
data/splits/window_splits_stride{N}.parquet   ← ticker, fechas, window_row, split
```

Join canónico con features: `window_row` (alineación 1:1). Alternativo: `(ticker, window_start_date, window_end_date)`.


## 2. Verificación SHA + contrato de fechas del manifest

Leemos `checksums.sha256` y comprobamos digests en disco.
Si `data_version != splits-0.1.0` o falla un SHA → **PARA**.


In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

checksums_path = SPLITS_DIR / "checksums.sha256"
manifest_path = SPLITS_DIR / "split_manifest.json"
assert checksums_path.is_file() and manifest_path.is_file()

expected = {}
for line in checksums_path.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if not line:
        continue
    digest, rel = line.split(None, 1)
    expected[rel.strip()] = digest

required_rels = [
    *[f"data/splits/window_splits_stride{s}.parquet" for s in EXPECTED_STRIDES],
]
missing = [r for r in required_rels if r not in expected]
if missing:
    raise RuntimeError(f"Faltan entradas en checksums.sha256: {missing}. PARAR.")

print("=== Verificación SHA (splits) ===")
sha_ok = True
for rel in required_rels:
    path = ROOT / rel
    if not path.is_file():
        print(f"  MISSING  {rel}")
        sha_ok = False
        continue
    got = sha256_file(path)
    exp = expected[rel]
    ok = got == exp
    sha_ok = sha_ok and ok
    status = "OK" if ok else "FAIL"
    print(f"  {status}  {rel}")
    print(f"         expected={exp}")
    print(f"         got     ={got}")
    if not ok:
        raise RuntimeError(f"SHA mismatch: {rel}. PARAR.")

if not sha_ok:
    raise RuntimeError("SHA verification failed. PARAR.")
print("\nTodos los SHA de window_splits_* OK.")


In [ ]:
with manifest_path.open(encoding="utf-8") as f:
    manifest = json.load(f)

version = manifest.get("data_version")
if version != EXPECTED_VERSION:
    raise RuntimeError(f"data_version={version!r} != {EXPECTED_VERSION!r}. PARAR.")

feat_ver = manifest.get("features_data_version")
if feat_ver != EXPECTED_FEATURES_VERSION:
    raise RuntimeError(f"features_data_version={feat_ver!r} != {EXPECTED_FEATURES_VERSION!r}. PARAR.")

primary = int(manifest.get("primary_stride", -1))
if primary != PRIMARY_STRIDE:
    raise RuntimeError(f"primary_stride={primary} != {PRIMARY_STRIDE}. PARAR.")

# Ficheros por stride deben existir
for s in EXPECTED_STRIDES:
    p = SPLITS_DIR / f"window_splits_stride{s}.parquet"
    if not p.is_file():
        raise RuntimeError(f"Falta parquet de splits: {p}. PARAR.")

print(f"data_version            : {version}")
print(f"features_data_version   : {feat_ver}")
print(f"built_at_utc            : {manifest.get('built_at_utc')}")
print(f"primary_stride          : {primary}")
print(f"donors ({len(manifest['donors'])}) : {', '.join(manifest['donors'])}")
print(f"target                  : {manifest['target']}")
print()
print("=== Contrato de fechas (window_end_date inclusive) ===")
rules = manifest["date_rules"]
rows = []
for label in SPLIT_ORDER:
    r = rules[label]
    if "window_end_date_inclusive" in r:
        lo, hi = r["window_end_date_inclusive"]
        rows.append({"split": label, "tickers": r["tickers"], "window_end_from": lo, "window_end_to": hi})
    else:
        rows.append({"split": label, "tickers": r["tickers"], "window_end_from": "(resto)", "window_end_to": r.get("note", "")})
display(pd.DataFrame(rows))


## 3. Tabla de conteos: split × stride

Cada fila del parquet de splits es una ventana. Los conteos **deben cuadrar** con
`n_windows_by_split_by_stride` del manifest.


In [ ]:
splits_by_stride = {}
for s in EXPECTED_STRIDES:
    path = SPLITS_DIR / f"window_splits_stride{s}.parquet"
    df = pd.read_parquet(path)
    # sanity schema
    assert list(df.columns) == manifest["schema"]["split_columns"], (
        f"schema stride{s}: {list(df.columns)} != {manifest['schema']['split_columns']}"
    )
    splits_by_stride[s] = df

# Tabla conteos observados
obs = pd.DataFrame({
    s: splits_by_stride[s]["split"].value_counts()
    for s in EXPECTED_STRIDES
}).reindex(SPLIT_ORDER).fillna(0).astype(int)
obs.columns = [f"stride{s}" for s in EXPECTED_STRIDES]
obs.index.name = "split"

# Tabla manifest
man = pd.DataFrame({
    s: manifest["n_windows_by_split_by_stride"][f"stride{s}"]
    for s in EXPECTED_STRIDES
}).reindex(SPLIT_ORDER).fillna(0).astype(int)
man.columns = [f"stride{s}" for s in EXPECTED_STRIDES]
man.index.name = "split"

print("--- Conteos observados (parquets) ---")
display(obs)
print("--- Conteos manifest ---")
display(man)

diff = obs - man
print("--- Diferencia (obs - manifest); debe ser todo 0 ---")
display(diff)
assert (diff == 0).all().all(), "Conteos NO cuadran con manifest. PARAR."
print("Conteos cuadran con manifest: SÍ")


## 4. Composición de splits (barras)

Stacked / barras: proporción de ventanas por etiqueta. Default `STRIDE=1`;
opcionalmente comparamos con `STRIDE_COMPARE`.


In [ ]:
def composition_frame(stride: int) -> pd.Series:
    vc = splits_by_stride[stride]["split"].value_counts().reindex(SPLIT_ORDER).fillna(0).astype(int)
    return vc

comp_a = composition_frame(STRIDE)
comp_b = composition_frame(STRIDE_COMPARE)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)

for ax, stride, comp in [
    (axes[0], STRIDE, comp_a),
    (axes[1], STRIDE_COMPARE, comp_b),
]:
    colors = [SPLIT_COLORS[s] for s in comp.index]
    bars = ax.bar(comp.index, comp.values, color=colors, edgecolor="white", linewidth=0.5)
    ax.set_title(f"Composición de splits — stride={stride}")
    ax.set_ylabel("n_windows")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(True, axis="y", alpha=0.3)
    for b, v in zip(bars, comp.values):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{v}",
                ha="center", va="bottom", fontsize=8)

fig.tight_layout()
if SAVE_FIGS:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    out = FIG_DIR / f"04b_composition_stride{STRIDE}_vs_{STRIDE_COMPARE}.png"
    fig.savefig(out, dpi=120)
    print(f"guardado: {out.relative_to(ROOT)}")
plt.show()

# Stacked horizontal (proporciones stride default)
fig2, ax2 = plt.subplots(figsize=(10, 1.8))
total = int(comp_a.sum())
left = 0.0
for label, n in comp_a.items():
    w = n / total
    ax2.barh(0, w, left=left, color=SPLIT_COLORS[label], edgecolor="white",
             label=f"{label} ({n})")
    if w >= 0.04:
        ax2.text(left + w / 2, 0, f"{100*w:.0f}%", ha="center", va="center",
                 fontsize=8, color="white", fontweight="bold")
    left += w
ax2.set_yticks([])
ax2.set_xlim(0, 1)
ax2.set_xlabel("proporción")
ax2.set_title(f"Proporción stacked — stride={STRIDE} (total={total})")
ax2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=3, fontsize=8, frameon=False)
fig2.tight_layout()
if SAVE_FIGS:
    out2 = FIG_DIR / f"04b_stacked_stride{STRIDE}.png"
    fig2.savefig(out2, dpi=120, bbox_inches="tight")
    print(f"guardado: {out2.relative_to(ROOT)}")
plt.show()


## 5. Timeline didáctica: rangos de `window_end_date` por split

Para `STRIDE` (default 1): min/max de `window_end_date` observados por etiqueta.
Las líneas verticales marcan los bordes del contrato del manifest.


In [ ]:
df_s = splits_by_stride[STRIDE].copy()
df_s["window_end_date"] = pd.to_datetime(df_s["window_end_date"])

ranges = (
    df_s.groupby("split")["window_end_date"]
    .agg(min_end="min", max_end="max", n="count")
    .reindex(SPLIT_ORDER)
)
print(f"=== Rangos window_end_date (stride={STRIDE}) ===")
display(ranges)

# Contract boundaries (manifest) as vertical refs
contract_bounds = []
for label, r in manifest["date_rules"].items():
    if "window_end_date_inclusive" in r:
        lo, hi = r["window_end_date_inclusive"]
        contract_bounds.append((label, pd.Timestamp(lo), pd.Timestamp(hi)))

fig, ax = plt.subplots(figsize=(11, 3.5))
y_pos = {lab: i for i, lab in enumerate(SPLIT_ORDER)}
for lab in SPLIT_ORDER:
    row = ranges.loc[lab]
    y = y_pos[lab]
    ax.barh(y, (row["max_end"] - row["min_end"]).days + 1,
            left=mdates.date2num(row["min_end"]),
            height=0.55, color=SPLIT_COLORS[lab], edgecolor="white",
            label=f"{lab} (n={int(row['n'])})")
    ax.text(mdates.date2num(row["min_end"]), y + 0.35,
            row["min_end"].strftime("%Y-%m-%d"), fontsize=7, va="bottom")
    ax.text(mdates.date2num(row["max_end"]), y + 0.35,
            row["max_end"].strftime("%Y-%m-%d"), fontsize=7, va="bottom", ha="right")

# Vertical markers for key contract starts
for ts, name in [
    (pd.Timestamp("2012-01-01"), "2012"),
    (pd.Timestamp("2022-01-01"), "2022"),
    (pd.Timestamp("2022-07-01"), "NVDA vis"),
    (pd.Timestamp("2023-01-01"), "test"),
]:
    ax.axvline(mdates.date2num(ts), color="black", ls="--", lw=0.7, alpha=0.5)
    ax.text(mdates.date2num(ts), len(SPLIT_ORDER) - 0.3, name,
            fontsize=7, rotation=90, va="top", ha="right", alpha=0.7)

ax.set_yticks(list(y_pos.values()))
ax.set_yticklabels(list(y_pos.keys()))
ax.xaxis_date()
ax.set_xlabel("window_end_date")
ax.set_title(f"Timeline de rangos por split — stride={STRIDE}")
ax.grid(True, axis="x", alpha=0.3)
fig.tight_layout()
if SAVE_FIGS:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    out = FIG_DIR / f"04b_timeline_stride{STRIDE}.png"
    fig.savefig(out, dpi=120)
    print(f"guardado: {out.relative_to(ROOT)}")
plt.show()


## 6. Assert visible: 0 NVDA en `donor_train` / `donor_val`

El generador se entrena solo con donors. NVDA **nunca** debe aparecer en train/val de donors.


In [ ]:
print("=== NVDA en donor_train ∪ donor_val (debe ser 0 en cada stride) ===")
rows = []
for s in EXPECTED_STRIDES:
    df = splits_by_stride[s]
    mask = df["split"].isin(["donor_train", "donor_val"]) & (df["ticker"] == "NVDA")
    n = int(mask.sum())
    rows.append({"stride": s, "n_nvda_in_donor_train_or_val": n})
    assert n == 0, f"stride={s}: hay {n} ventanas NVDA en donor_train/val. PARAR."
    # refuerzo: tickers en donor_* solo donors
    donors = set(manifest["donors"])
    donor_mask = df["split"].isin(["donor_train", "donor_val"])
    tickers = set(df.loc[donor_mask, "ticker"].unique())
    assert tickers <= donors, f"stride={s}: tickers extra en donor_*: {tickers - donors}"

display(pd.DataFrame(rows))
print("Assert 0 NVDA en donor_train/val: OK (todos los strides)")


## 7. Demo join: splits + windows → filtrar `donor_train`

Patrón que usará un compañero al entrenar:

1. Cargar `window_splits_stride{N}.parquet` y `windows_65_stride{N}.parquet`
2. Asignar `window_row = 0..n-1` en windows (alineación 1:1 por posición) y join con splits
3. Filtrar `split == "donor_train"` (entrenamiento) o `"donor_val"` (checkpoint)

Join alternativo: `(ticker, window_start_date, window_end_date)`.

Aquí demo con **stride=1** (SHA del windows verificado).


In [ ]:
# Demo join con stride=1 (SHA del windows verificado en el brief).
# windows_* no trae window_row: el contrato es alineación 1:1 por posición de fila.
JOIN_STRIDE = 1
windows_path = FEATURES_DIR / f"windows_65_stride{JOIN_STRIDE}.parquet"
assert windows_path.is_file(), f"Falta {windows_path} para demo join"

got_w = sha256_file(windows_path)
if got_w != WINDOWS_STRIDE1_SHA:
    raise RuntimeError(
        f"SHA windows_65_stride1 mismatch.\n"
        f"  expected={WINDOWS_STRIDE1_SHA}\n  got={got_w}\nPARAR."
    )
print(f"SHA windows_65_stride{JOIN_STRIDE}: OK ({got_w[:16]}…)")

splits_j = splits_by_stride[JOIN_STRIDE].copy()
windows = pd.read_parquet(windows_path).copy()
print(f"splits shape : {splits_j.shape}")
print(f"windows shape: {windows.shape}  cols={list(windows.columns)}")

# 1) Join canónico: window_row = índice de fila (0..n-1), alineado 1:1
windows["window_row"] = np.arange(len(windows), dtype=np.int64)
assert (windows["window_row"].values == splits_j["window_row"].values).all()
# refuerzo: claves naturales alineadas
assert (windows["ticker"].astype(str).values == splits_j["ticker"].astype(str).values).all()

merged = windows.merge(
    splits_j[["window_row", "split"]],
    on="window_row",
    how="inner",
    validate="1:1",
)
assert len(merged) == len(windows) == len(splits_j), "Join no es 1:1. PARAR."
print(f"merged shape : {merged.shape}  (1:1 OK vía window_row)")

donor_train = merged.loc[merged["split"] == "donor_train"]
print(
    f"\ndonor_train: {len(donor_train)} ventanas "
    f"(manifest={manifest['n_windows_by_split_by_stride']['stride1']['donor_train']})"
)
assert len(donor_train) == manifest["n_windows_by_split_by_stride"]["stride1"]["donor_train"]

print("\n--- donor_train head (cómo filtrar para entrenar) ---")
show_cols = ["ticker", "window_start_date", "window_end_date", "window_row", "split"]
display(donor_train[show_cols].head(8))

print("\nSnippet típico para un compañero:")
print("""
splits = pd.read_parquet("data/splits/window_splits_stride1.parquet")
windows = pd.read_parquet("data/features/windows_65_stride1.parquet").copy()
windows["window_row"] = range(len(windows))  # alineación 1:1 por posición
df = windows.merge(splits[["window_row", "split"]], on="window_row", how="inner", validate="1:1")
train = df.loc[df["split"] == "donor_train"]   # entrenar generador
val   = df.loc[df["split"] == "donor_val"]     # checkpoint / early-stop
# test = df.loc[df["split"] == "nvda_test"]    # SOLO evaluación final — no tunear
""")


## 8. Advertencia: stride alto ⇒ `nvda_visible` muy pequeño

Las reglas de fecha son las mismas en todos los strides, pero al subir el stride
hay **menos ventanas** que caen en la ventana corta de NVDA visible (H2-2022).


In [ ]:
vis = pd.DataFrame({
    "stride": EXPECTED_STRIDES,
    "nvda_visible": [
        manifest["n_windows_by_split_by_stride"][f"stride{s}"]["nvda_visible"]
        for s in EXPECTED_STRIDES
    ],
    "nvda_test": [
        manifest["n_windows_by_split_by_stride"][f"stride{s}"]["nvda_test"]
        for s in EXPECTED_STRIDES
    ],
    "donor_train": [
        manifest["n_windows_by_split_by_stride"][f"stride{s}"]["donor_train"]
        for s in EXPECTED_STRIDES
    ],
})
# verificar vs parquet
for s in EXPECTED_STRIDES:
    obs_v = int((splits_by_stride[s]["split"] == "nvda_visible").sum())
    man_v = int(manifest["n_windows_by_split_by_stride"][f"stride{s}"]["nvda_visible"])
    assert obs_v == man_v

display(vis)

fig, ax = plt.subplots(figsize=(7, 3.5))
x = np.arange(len(EXPECTED_STRIDES))
w = 0.35
ax.bar(x - w/2, vis["nvda_visible"], width=w, color=SPLIT_COLORS["nvda_visible"], label="nvda_visible")
ax.bar(x + w/2, vis["nvda_test"], width=w, color=SPLIT_COLORS["nvda_test"], label="nvda_test")
ax.set_xticks(x)
ax.set_xticklabels([f"stride={s}" for s in EXPECTED_STRIDES])
ax.set_ylabel("n_windows")
ax.set_title("nvda_visible vs nvda_test por stride")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
for i, (a, b) in enumerate(zip(vis["nvda_visible"], vis["nvda_test"])):
    ax.text(i - w/2, a, str(a), ha="center", va="bottom", fontsize=8)
    ax.text(i + w/2, b, str(b), ha="center", va="bottom", fontsize=8)
fig.tight_layout()
if SAVE_FIGS:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    out = FIG_DIR / "04b_nvda_visible_by_stride.png"
    fig.savefig(out, dpi=120)
    print(f"guardado: {out.relative_to(ROOT)}")
plt.show()

print("⚠️  Con stride=65, nvda_visible tiene solo "
      f"{int(vis.loc[vis['stride']==65, 'nvda_visible'].iloc[0])} ventanas. "
      "Cuidado al usarlo para calibración / inspección.")


## 9. Cierre — handoff a generadores

### Protocolo de uso

| Qué | Split | Notas |
|---|---|---|
| Entrenar generador | `donor_train` | Solo donors 2012–2021 |
| Checkpoint / early-stop | `donor_val` | Solo donors 2022 |
| Inspección / calibración NVDA | `nvda_visible` | **Fuera** del train del generador |
| Evaluación final | `nvda_test` | **No mirar para tunear** |
| No usar | `unused` | Fuera del protocolo oficial |

### Checklist antes de entrenar

1. Elegir un `stride` ∈ {1, 10, 30, 65} y usar **el mismo** en features + splits.
2. Verificar SHA de `window_splits_stride{N}.parquet` (`checksums.sha256`).
3. Join por `window_row` → filtrar `donor_train` (+ `donor_val` para checkpoint).
4. No filtrar ni mirar `nvda_test` durante el desarrollo / tuning.

Fuente de verdad: `data/splits/` (`splits-0.1.0`), no este notebook.


In [ ]:
print("=" * 60)
print("04b inspect splits — resumen")
print("=" * 60)
print(f"data_version     : {manifest['data_version']}")
print(f"SHA splits       : OK (stride 1/10/30/65)")
print(f"Conteos=manifest : SÍ")
print(f"0 NVDA en train/val donors: OK")
print()
print("nvda_visible por stride:")
for s in EXPECTED_STRIDES:
    n = manifest["n_windows_by_split_by_stride"][f"stride{s}"]["nvda_visible"]
    print(f"  stride={s:>2}: {n}")
print()
print("Handoff generadores: SÍ")
print("  → entrenar solo con donor_train (+ donor_val para checkpoint)")
print("  → no mirar nvda_test para tunear")
print("=" * 60)
